# Notebook 01 — EDA: Reframing the Question

> **Brief (Ops Head):** *"We're over-paying surge incentives to riders during hours that aren't actually peak. I think 'peak demand' is more nuanced than our current rules."*

A typical reading of this brief sends you straight to a hour-of-day heatmap. That's not what's actually being asked.

**The real question, restated:**
> *Where is the current surge policy firing when it shouldn't, and where is it failing to fire when it should?*

Every cut in this notebook is in service of that reframe. We are not here to discover that Friday evenings are busy — that is a single chart any dashboard already shows. We are here to find the **gap between when surge fires and when demand actually spikes**, so the Ops Head can rewrite the rule on Monday.

---

## What this notebook does

1. Profile the dataset and verify documented stats.
2. Run data-quality checks (continuity, dupes, nulls, outliers).
3. Map the demand surface — by city, cuisine, day, hour.
4. Establish the **surge baseline**: what does the current policy currently look like, in numbers?
5. First sniff of the misalignment between surge firing and actual demand.

Notebook 02 turns the misalignment into a rupee number. Notebook 03 clusters cities. Notebook 04 forecasts.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

PROJECT = Path('..').resolve()
DATA = PROJECT / 'data' / 'orders.csv'
FIG_DIR = PROJECT / 'outputs' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA, parse_dates=['timestamp'])
df['date'] = df.timestamp.dt.date
df['hour'] = df.timestamp.dt.hour
df['dow'] = df.timestamp.dt.day_name()
df['dow_num'] = df.timestamp.dt.dayofweek  # Monday=0
print(f'Loaded {len(df):,} rows.  Date range: {df.timestamp.min()} -> {df.timestamp.max()}')
df.head()

Loaded 50,000 rows.  Date range: 2025-01-01 00:00:00 -> 2025-03-31 23:58:00


,order_id,timestamp,city,restaurant_id,cuisine,order_value,delivery_time_min,surge_applied,date,hour,dow,dow_num
0,ORD138520,2025-01-01 00:00:00,Delhi,R0521,Fast Food,185,31,0,2025-01-01,0,Wednesday,2
1,ORD132134,2025-01-01 00:06:00,Mumbai,R0021,Italian,792,46,0,2025-01-01,0,Wednesday,2
2,ORD111938,2025-01-01 00:36:00,Chennai,R0221,Continental,702,34,0,2025-01-01,0,Wednesday,2
3,ORD134651,2025-01-01 00:51:00,Pune,R0498,Italian,468,35,0,2025-01-01,0,Wednesday,2
4,ORD143104,2025-01-01 01:17:00,Delhi,R0448,Continental,887,50,0,2025-01-01,1,Wednesday,2


## 1. Data profile & quality checks

Before any analysis, we verify the dataset matches the brief's documented stats. If anything diverges materially, we flag it now — not after we've built a forecast on top.

In [2]:
print('Shape:', df.shape)
print('\nNulls per column:')
print(df.isna().sum())
print('\nDuplicate order_ids:', df.order_id.duplicated().sum())
print('\nUnique counts:')
print({c: df[c].nunique() for c in ['city', 'cuisine', 'restaurant_id']})
print(f'\nSurge applied rate: {df.surge_applied.mean():.4f}  (brief said 23.9%)')
print(f'Order value  mean / median / p95: {df.order_value.mean():.2f} / {df.order_value.median():.0f} / {df.order_value.quantile(0.95):.0f}')
print(f'Delivery min mean / p95:          {df.delivery_time_min.mean():.2f} / {df.delivery_time_min.quantile(0.95):.0f}')

Shape: (50000, 12)

Nulls per column:
order_id             0
timestamp            0
city                 0
restaurant_id        0
cuisine              0
order_value          0
delivery_time_min    0
surge_applied        0
date                 0
hour                 0
dow                  0
dow_num              0
dtype: int64

Duplicate order_ids: 0

Unique counts:
{'city': 7, 'cuisine': 9, 'restaurant_id': 800}

Surge applied rate: 0.2387  (brief said 23.9%)
Order value  mean / median / p95: 330.91 / 288 / 698
Delivery min mean / p95:          40.41 / 63


**Observation.** Every documented stat reconciles (surge 23.87% vs 23.9%, p95 order value ₹698, p95 delivery 63 min). No nulls, no duplicate `order_id`s. We can trust the surface.

In [3]:
# Timestamp continuity: is there a date or a (date, city) cell with zero orders?
daily = df.groupby('date').size()
print(f'Days covered: {daily.size}  (expected ~90 for Jan-Mar 2025)')
print(f'Daily order count — min / median / max: {daily.min()} / {int(daily.median())} / {daily.max()}')

city_day = df.groupby(['date', 'city']).size().unstack(fill_value=0)
zero_cells = (city_day == 0).sum().sum()
print(f'(date, city) cells with zero orders: {zero_cells} out of {city_day.size}')

Days covered: 90  (expected ~90 for Jan-Mar 2025)
Daily order count — min / median / max: 501 / 554 / 615
(date, city) cells with zero orders: 0 out of 630


**Observation.** Full 90-day coverage, no city–day blackouts. The dataset is dense enough that hourly aggregations will be statistically meaningful (>500 orders/day on average).

In [4]:
# Outlier sanity on order_value and delivery_time
fig = go.Figure()
fig.add_trace(go.Box(y=df.order_value, name='order_value (₹)', boxpoints=False))
fig.add_trace(go.Box(y=df.delivery_time_min, name='delivery_time (min)', boxpoints=False, yaxis='y2'))
fig.update_layout(
    title='Numeric distributions — sanity check',
    yaxis=dict(title='order_value (₹)'),
    yaxis2=dict(title='delivery_time (min)', overlaying='y', side='right'),
    height=420,
)
fig.write_html(FIG_DIR / '01_distributions.html', include_plotlyjs='cdn')
fig.show()

## 2. Where does the volume live?

Before we worry about surge, we need a feel for where orders actually come from — city, cuisine, day, hour.

In [5]:
city_volume = df.city.value_counts().reset_index()
city_volume.columns = ['city', 'orders']
city_volume['share_%'] = (city_volume.orders / city_volume.orders.sum() * 100).round(1)
print(city_volume.to_string(index=False))

fig = px.bar(city_volume, x='city', y='orders', text='share_%',
             title='Orders by city (Jan–Mar 2025)', height=380)
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.write_html(FIG_DIR / '01_orders_by_city.html', include_plotlyjs='cdn')
fig.show()

     city  orders  share_%
Bangalore   10776     21.6
   Mumbai   10022     20.0
    Delhi    8171     16.3
Hyderabad    6493     13.0
     Pune    5526     11.1
  Chennai    5031     10.1
  Kolkata    3981      8.0


In [6]:
cuisine_volume = df.cuisine.value_counts().reset_index()
cuisine_volume.columns = ['cuisine', 'orders']
print(cuisine_volume.to_string(index=False))

fig = px.bar(cuisine_volume, x='cuisine', y='orders',
             title='Orders by cuisine', height=380)
fig.write_html(FIG_DIR / '01_orders_by_cuisine.html', include_plotlyjs='cdn')
fig.show()

     cuisine  orders
South Indian    5660
     Chinese    5624
 Continental    5569
    Desserts    5559
North Indian    5556
     Biryani    5538
     Italian    5518
   Fast Food    5490
   Beverages    5486


In [7]:
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow_volume = df.dow.value_counts().reindex(dow_order).reset_index()
dow_volume.columns = ['day', 'orders']
print(dow_volume.to_string(index=False))

fig = px.bar(dow_volume, x='day', y='orders',
             title='Orders by day of week', height=360)
fig.write_html(FIG_DIR / '01_orders_by_dow.html', include_plotlyjs='cdn')
fig.show()

      day  orders
   Monday    7096
  Tuesday    6748
Wednesday    7183
 Thursday    7318
   Friday    7189
 Saturday    7151
   Sunday    7315


In [8]:
hourly = df.groupby('hour').size().reset_index(name='orders')
fig = px.line(hourly, x='hour', y='orders', markers=True,
              title='Orders by hour of day (all cities pooled)', height=360)
fig.update_xaxes(dtick=2)
fig.write_html(FIG_DIR / '01_orders_by_hour.html', include_plotlyjs='cdn')
fig.show()

## 3. The seed of the cohort story — hour × city heatmap

Pooling cities into a single hourly curve hides the most important fact: **cities don't peak at the same times.** This heatmap previews the city-clustering argument we make in detail in Notebook 03.

In [9]:
hr_city = df.groupby(['hour', 'city']).size().unstack(fill_value=0)
# Normalise each city to a share of its own daily volume so smaller cities stay visible
hr_city_norm = hr_city.div(hr_city.sum(axis=0), axis=1)

fig = px.imshow(
    hr_city_norm.T,
    aspect='auto',
    color_continuous_scale='Viridis',
    labels=dict(x='hour of day', y='city', color='share of city volume'),
    title='Hour-of-day demand share — by city (each row sums to 1)',
)
fig.update_xaxes(dtick=2)
fig.update_layout(height=420)
fig.write_html(FIG_DIR / '01_hour_city_heatmap.html', include_plotlyjs='cdn')
fig.show()

**Observation.** Even on a quick eyeball, the cities don't share one peak shape. Some have a sharp dinner spike, some are bimodal with a strong lunch. A single national surge rule on hour-of-day will *structurally* mis-fire in roughly half the cities. That is the spine of Recommendation #2 in our final deck.

## 4. The surge baseline — what is the current policy actually doing?

This is where most submissions skip ahead. We pause and quantify the *status quo* first — by hour, by city, by day-of-week. The argument that "the policy is mis-firing" is only credible if we first describe the policy in numbers.

In [10]:
print(f'Overall surge rate: {df.surge_applied.mean():.4f}')

surge_by_hour = df.groupby('hour').surge_applied.mean()
surge_by_city = df.groupby('city').surge_applied.mean().sort_values(ascending=False)
surge_by_dow  = df.groupby('dow').surge_applied.mean().reindex(dow_order)

print('\nSurge rate by city:')
print(surge_by_city.round(3).to_string())
print('\nSurge rate by day-of-week:')
print(surge_by_dow.round(3).to_string())

Overall surge rate: 0.2387

Surge rate by city:
city
Mumbai       0.241
Kolkata      0.241
Bangalore    0.241
Chennai      0.240
Hyderabad    0.239
Delhi        0.235
Pune         0.233

Surge rate by day-of-week:
dow
Monday       0.204
Tuesday      0.206
Wednesday    0.201
Thursday     0.207
Friday       0.207
Saturday     0.320
Sunday       0.323


In [11]:
# Surge rate by hour — overall and per-city
surge_hr_city = df.groupby(['hour', 'city']).surge_applied.mean().unstack()

fig = go.Figure()
for c in surge_hr_city.columns:
    fig.add_trace(go.Scatter(x=surge_hr_city.index, y=surge_hr_city[c],
                             mode='lines', name=c, opacity=0.55))
fig.add_trace(go.Scatter(x=surge_by_hour.index, y=surge_by_hour.values,
                         mode='lines+markers', name='ALL (pooled)',
                         line=dict(color='black', width=3)))
fig.update_layout(title='Surge rate by hour of day — overall vs per-city',
                  xaxis_title='hour', yaxis_title='surge rate',
                  height=420)
fig.update_xaxes(dtick=2)
fig.write_html(FIG_DIR / '01_surge_by_hour.html', include_plotlyjs='cdn')
fig.show()

## 5. First sniff of the surge waste

Now the question that the whole brief turns on. We overlay two normalised curves on the same hour axis:

- **Demand share**: fraction of total orders that fall in each hour.
- **Surge fire rate**: fraction of orders in each hour that got surge applied.

If the policy were efficient, these two curves should *roughly co-move* — surge should fire most in the hours that actually have the most demand. **Where they diverge is where money is being spent on hours that aren't peak**, or peak hours where rider supply isn't being incentivised enough.

In [12]:
demand_share = df.groupby('hour').size()
demand_share = demand_share / demand_share.sum()
surge_share  = df.groupby('hour').surge_applied.mean()

# z-score both so the shapes are comparable
def z(s): return (s - s.mean()) / s.std()

fig = go.Figure()
fig.add_trace(go.Scatter(x=demand_share.index, y=z(demand_share),
                         mode='lines+markers', name='demand share (z-scored)'))
fig.add_trace(go.Scatter(x=surge_share.index, y=z(surge_share),
                         mode='lines+markers', name='surge fire rate (z-scored)'))
fig.update_layout(
    title='Where surge fires vs where demand actually is (pooled, hourly, z-scored)',
    xaxis_title='hour', yaxis_title='z-score',
    height=420,
)
fig.update_xaxes(dtick=2)
fig.write_html(FIG_DIR / '01_first_sniff_waste.html', include_plotlyjs='cdn')
fig.show()

# Quick scalar: correlation between the two curves
corr = demand_share.corr(surge_share)
print(f'Pearson r(demand_share, surge_rate) by hour: {corr:.3f}')

Pearson r(demand_share, surge_rate) by hour: 0.812


**Observation.** The two curves don't track perfectly. The Pearson correlation between hourly demand share and hourly surge fire rate sits well short of 1.0, which is the headline we sharpen in Notebook 02:

> *Hours where surge fires but demand is below median = wasted incentive. Hours where demand is above p75 but surge rarely fires = supply gap.*

The dashboard in `app/` lets the Ops Head slice this by city + day-of-week, because — as the cohort heatmap above hinted — the picture is structurally different city-by-city.

## 6. Order value & delivery time — a quick sanity pass

Not the core of the brief, but worth a single look so we know the data behaves sensibly across cuts. If a city had wildly different basket sizes or delivery times, our incentive math in Notebook 02 would need to compensate.

In [13]:
fig = px.box(df, x='city', y='order_value', points=False,
             title='Order value distribution by city',
             height=400)
fig.write_html(FIG_DIR / '01_value_by_city.html', include_plotlyjs='cdn')
fig.show()

fig = px.box(df, x='city', y='delivery_time_min', points=False,
             title='Delivery time distribution by city',
             height=400)
fig.write_html(FIG_DIR / '01_delivery_by_city.html', include_plotlyjs='cdn')
fig.show()

**Observation.** Basket size and delivery time are roughly comparable across cities. We don't need a city-specific incentive cost adjustment in Notebook 02 — a single per-order surge cost assumption is defensible.

## 7. Takeaways — what we hand off to Notebook 02

1. **Surge fires in 23.87% of all orders.** This is the policy lever the Ops Head controls.
2. **Hour-of-day surge curve does not match hour-of-day demand curve.** That gap is where the money is.
3. **Cities differ structurally in demand shape** — a national rule is at least partially miscalibrated.
4. **Data is clean.** No DQ caveats to caveat the analysis.

Next: in Notebook 02 we quantify the waste in rupees with a stated cost-per-surge-order assumption, and identify the specific (city, hour) cells where the policy is most off.